In [2]:
import os
import urllib.request
import pandas as pd

os.makedirs("../data/SARS2_RBD", exist_ok=True)

url = "https://media.githubusercontent.com/media/jbloomlab/SARS2_RBD_Ab_escape_maps/refs/heads/main/processed_data/escape_data.csv"
out_path = "../data/SARS2_RBD/escape_data.csv"

if not os.path.exists(out_path):
    urllib.request.urlretrieve(url, out_path)
    print("Downloaded")
else:
    print("Already exists")

df = pd.read_csv(out_path)
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())
print("\nFirst 5 rows:")
print(df.head())

Downloaded


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_27860\2202023030.py:16: DtypeWarning: Columns (0: condition_alias) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(out_path)


Shape: (392456, 18)

Columns:
['condition', 'condition_type', 'condition_subtype', 'condition_year', 'condition_alias', 'eliciting_virus', 'known_to_neutralize', 'IC50s', 'site', 'wildtype', 'mutation', 'mut_escape', 'study', 'lab', 'site_total_escape', 'label_site', 'protein_site', 'protein_chain']

First 5 rows:
   condition condition_type condition_subtype  condition_year condition_alias  \
0  COV2-2196       antibody           class 1            2020       (AZD8895)   
1  COV2-2196       antibody           class 1            2020       (AZD8895)   
2  COV2-2196       antibody           class 1            2020       (AZD8895)   
3  COV2-2196       antibody           class 1            2020       (AZD8895)   
4  COV2-2196       antibody           class 1            2020       (AZD8895)   

                     eliciting_virus known_to_neutralize  IC50s  site  \
0  SARS-CoV-2;pre-Omicron SARS-CoV-2      Wuhan-Hu-1;any  NA;NA   331   
1  SARS-CoV-2;pre-Omicron SARS-CoV-2      Wuhan-Hu-

In [3]:
df = df[df['condition_type'] == 'antibody'].copy()
df = df[df['mut_escape'].notna()].copy()
df = df[df['mutation'].notna()].copy()

df = df.rename(columns={
    'condition': 'antibody',
    'mutation': 'mutant',
    'mut_escape': 'escape_mean'
})

df['label'] = (df['escape_mean'] > 0.5).astype(int)

print("Rows:", len(df))
print("Antibodies:", df['antibody'].nunique())
print("Positive rate:", df['label'].mean())
print("Sites:", df['site'].min(), "to", df['site'].max())

Rows: 269128
Antibodies: 1535
Positive rate: 0.06344936238518474
Sites: 331 to 531


In [4]:
# Check study distribution
print("Studies:")
print(df['study'].value_counts().head(20))

print("\nCondition subtypes:")
print(df['condition_subtype'].value_counts().head(20))

print("\nMutations per antibody (top 20):")
print(df.groupby('antibody').size().sort_values(ascending=False).head(20))

print("\nAntibodies with >= 1000 mutations:")
good_abs = df.groupby('antibody').size()
good_abs = good_abs[good_abs >= 1000].index.tolist()
print(f"Count: {len(good_abs)}")

Studies:
study
2022_Cao_BA2-4-5            168507
2022_Cao_Omicron             30658
2021_Starr_Vir               23365
2021_Greaney_Crowe_Abs       19337
2021_Greaney_Rockefeller     11699
2021_Starr_REGN               5829
2021_Dong_AZ                  3897
2021_Tortorici_S2X259         1946
2021_Greaney_COV2-2955        1945
2021_Starr_LY-CoV555          1945
Name: count, dtype: int64

Condition subtypes:
condition_subtype
class 3    91070
class 1    69233
class 4    65680
class 2    43145
Name: count, dtype: int64

Mutations per antibody (top 20):
antibody
C110         2328
LY-CoV555    2290
REGN10933    2284
COV2-2196    2259
C121         2251
COVOX-222    2237
C144         2232
REGN10987    2220
COV2-2130    2189
LY-CoV016    2179
S304         2173
C002         2163
C105         2153
S2E12        2135
COV2-2499    2125
S2H13        2107
COV2-2955    2100
COVOX-150    2097
COV2-2479    2070
S309         2055
dtype: int64

Antibodies with >= 1000 mutations:
Count: 39


In [5]:
# Look at antibodies in the two candidate studies
for study_name in ['2021_Greaney_Crowe_Abs', '2021_Starr_Vir']:
    sub = df[df['study'] == study_name]
    print(f"\n=== {study_name} ===")
    print(f"Rows: {len(sub)}")
    print(f"Antibodies: {sub['antibody'].nunique()}")
    print(f"Sites: {sub['site'].min()} to {sub['site'].max()}")
    print(f"Positive rate: {(sub['escape_mean'] > 0.5).mean():.4f}")
    print("\nAntibody list:")
    print(sub.groupby('antibody').size().sort_values(ascending=False))
    print("\nSubtype distribution:")
    print(sub['condition_subtype'].value_counts())


=== 2021_Greaney_Crowe_Abs ===
Rows: 19337
Antibodies: 10
Sites: 331 to 531
Positive rate: 0.0013

Antibody list:
antibody
COV2-2499    1956
COV2-2677    1956
COV2-2096    1952
COV2-2094    1943
COV2-2479    1940
COV2-2082    1935
COV2-2832    1935
COV2-2165    1935
COV2-2050    1917
CR3022       1868
dtype: int64

Subtype distribution:
condition_subtype
class 4    7702
class 2    5809
class 1    3870
class 3    1956
Name: count, dtype: int64

=== 2021_Starr_Vir ===
Rows: 23365
Antibodies: 12
Sites: 331 to 531
Positive rate: 0.0307

Antibody list:
antibody
S2E12     1951
S2H13     1951
S2H14     1951
S309      1951
S2X58     1951
S304      1951
S2X35     1951
S2D106    1948
S2X16     1948
S2X227    1948
S2H58     1948
S2H97     1916
dtype: int64

Subtype distribution:
condition_subtype
class 2    9746
class 4    5818
class 1    3902
class 3    3899
Name: count, dtype: int64


In [7]:
for study_name in ['2021_Greaney_Crowe_Abs', '2021_Starr_Vir', 
                   '2021_Starr_REGN', '2022_Cao_Omicron',
                   '2021_Greaney_Rockefeller', '2021_Dong_AZ']:
    sub = df[df['study'] == study_name]
    if len(sub) == 0:
        continue
    pr_05 = (sub['escape_mean'] > 0.5).mean()
    pr_top10 = (sub['escape_mean'] > sub['escape_mean'].quantile(0.9)).mean()
    pr_top5 = (sub['escape_mean'] > sub['escape_mean'].quantile(0.95)).mean()
    print(f"{study_name:35s}  n={len(sub):>6}  abs={sub['antibody'].nunique():>3}  "
          f"PR@0.5={pr_05:.4f}  PR@top10%={pr_top10:.4f}  PR@top5%={pr_top5:.4f}")

2021_Greaney_Crowe_Abs               n= 19337  abs= 10  PR@0.5=0.0013  PR@top10%=0.1000  PR@top5%=0.0500
2021_Starr_Vir                       n= 23365  abs= 12  PR@0.5=0.0307  PR@top10%=0.0998  PR@top5%=0.0500
2021_Starr_REGN                      n=  5829  abs=  3  PR@0.5=0.0494  PR@top10%=0.1000  PR@top5%=0.0501
2022_Cao_Omicron                     n= 30658  abs=247  PR@0.5=0.0997  PR@top10%=0.1000  PR@top5%=0.0500
2021_Greaney_Rockefeller             n= 11699  abs=  6  PR@0.5=0.0322  PR@top10%=0.1000  PR@top5%=0.0500
2021_Dong_AZ                         n=  3897  abs=  2  PR@0.5=0.0290  PR@top10%=0.1001  PR@top5%=0.0500


In [8]:
study = '2021_Greaney_Crowe_Abs'
sub = df[df['study'] == study]
print(sub['escape_mean'].describe())
print("\nPercentiles:")
for p in [50, 75, 90, 95, 99]:
    print(f"  {p}th: {sub['escape_mean'].quantile(p/100):.4f}")

count    19337.000000
mean         0.007229
std          0.039233
min          0.000002
25%          0.000794
50%          0.001734
75%          0.002404
max          0.777200
Name: escape_mean, dtype: float64

Percentiles:
  50th: 0.0017
  75th: 0.0024
  90th: 0.0038
  95th: 0.0100
  99th: 0.1900


In [10]:
for study_name in ['2021_Starr_Vir', '2021_Starr_REGN', 
                   '2021_Greaney_Rockefeller', '2021_Dong_AZ']:
    sub = df[df['study'] == study_name]
    print(f"\n=== {study_name} ===")
    print(f"Rows: {len(sub)}, Antibodies: {sub['antibody'].nunique()}")
    ab_counts = sub.groupby('antibody').size().sort_values(ascending=False)
    print("Mutations per antibody:")
    print(ab_counts)
    print("\nClass distribution (top 30 abs):")
    top_abs = ab_counts.head(30).index.tolist()
    print(sub[sub['antibody'].isin(top_abs)]['condition_subtype'].value_counts())


=== 2021_Starr_Vir ===
Rows: 23365, Antibodies: 12
Mutations per antibody:
antibody
S2E12     1951
S2H13     1951
S2H14     1951
S309      1951
S2X58     1951
S304      1951
S2X35     1951
S2D106    1948
S2X16     1948
S2X227    1948
S2H58     1948
S2H97     1916
dtype: int64

Class distribution (top 30 abs):
condition_subtype
class 2    9746
class 4    5818
class 1    3902
class 3    3899
Name: count, dtype: int64

=== 2021_Starr_REGN ===
Rows: 5829, Antibodies: 3
Mutations per antibody:
antibody
REGN10987    1948
REGN10933    1943
LY-CoV016    1938
dtype: int64

Class distribution (top 30 abs):
condition_subtype
class 1    3881
class 3    1948
Name: count, dtype: int64

=== 2021_Greaney_Rockefeller ===
Rows: 11699, Antibodies: 6
Mutations per antibody:
antibody
C002    1950
C105    1950
C110    1950
C121    1950
C135    1950
C144    1949
dtype: int64

Class distribution (top 30 abs):
condition_subtype
class 2    5849
class 3    3900
class 1    1950
Name: count, dtype: int64

=== 202

In [11]:
study = '2021_Starr_Vir'
sub = df[df['study'] == study].copy()

# Class per antibody
print(sub.groupby(['antibody', 'condition_subtype']).size())

antibody  condition_subtype
S2D106    class 2              1948
S2E12     class 1              1951
S2H13     class 2              1951
S2H14     class 1              1951
S2H58     class 2              1948
S2H97     class 4              1916
S2X16     class 2              1948
S2X227    class 3              1948
S2X35     class 4              1951
S2X58     class 2              1951
S304      class 4              1951
S309      class 3              1951
dtype: int64


In [12]:
study = '2021_Starr_Vir'
df_s = df[df['study'] == study].copy()
df_s['label'] = (df_s['escape_mean'] > 0.5).astype(int)

print("Rows:", len(df_s))
print("Antibodies:", df_s['antibody'].nunique())
print("Positive rate:", df_s['label'].mean())
print("Sites:", df_s['site'].min(), "to", df_s['site'].max())
print("Missing sites:", set(range(df_s['site'].min(), df_s['site'].max()+1)) - set(df_s['site'].unique()))

Rows: 23365
Antibodies: 12
Positive rate: 0.030729723946073186
Sites: 331 to 531
Missing sites: {398, 400, 412, 416, 422, 423, 426, 431, 432, 436, 442, 454, 457, 467, 343, 350, 480, 355, 488, 491, 364, 495, 497, 502, 379, 509, 511}


In [13]:
rbd_seq_df = df_s.groupby('site')['wildtype'].agg(lambda x: x.mode()[0]).sort_index()
rbd_seq = ''.join(rbd_seq_df.values)
print("Length:", len(rbd_seq))
print("First 60:", rbd_seq[:60])
print("Last 60:", rbd_seq[-60:])

Length: 174
First 60: NITNLCPFGEVFATRFASYAWNKRISNCVAYSVLYNSASFSTFKYGVSPTKLNDLCFTNV
Last 60: PFERISTEIYQAGSTPNGVEGFNYFLQSGQPTNVGYQPYVVLSFELLHAPATVCGPKKST


In [14]:
import os
os.makedirs("../data/SARS2_RBD", exist_ok=True)

df_s.to_csv("../data/SARS2_RBD/starr_vir_features.csv", index=False)
with open("../data/SARS2_RBD/rbd_sequence.txt", "w") as f:
    f.write(rbd_seq)

print("Saved.")

Saved.


In [16]:
rbd_ref = "NITNLCPFGEVFNATRFASVYAWNRKRISNCVADYSVLYNSASFSTFKCYGVSPTKLNDLCFTNVYADSFVIRGDEVRQIAPGQTGKIADYNYKLPDDFTGCVIAWNSNNLDSKVGGNYNYLYRLFRKSNLKPFERDISTEIYQAGSTPCNGVEGFNCYFPLQSYGFQPTNGVGYQPYRVVVLSFELLHAPATVCGPKKST"

present = df_s.groupby('site')['wildtype'].first()
matches = 0
mismatches = []
for s, wt in present.items():
    if rbd_ref[s - 331] == wt:
        matches += 1
    else:
        mismatches.append((s, wt, rbd_ref[s-331]))

print(f"Matches: {matches} / {len(present)}")
if mismatches:
    print("Mismatches (site, data_wt, ref_wt):")
    for m in mismatches[:10]:
        print(m)

Matches: 174 / 174


In [17]:
import numpy as np
import torch
import esm
import os

# Full RBD reference (201 aa)
rbd_ref = "NITNLCPFGEVFNATRFASVYAWNRKRISNCVADYSVLYNSASFSTFKCYGVSPTKLNDLCFTNVYADSFVIRGDEVRQIAPGQTGKIADYNYKLPDDFTGCVIAWNSNNLDSKVGGNYNYLYRLFRKSNLKPFERDISTEIYQAGSTPCNGVEGFNCYFPLQSYGFQPTNGVGYQPYRVVVLSFELLHAPATVCGPKKST"
print("RBD length:", len(rbd_ref))

os.makedirs("../data/SARS2_RBD/esm_cache", exist_ok=True)
with open("../data/SARS2_RBD/rbd_sequence.txt", "w") as f:
    f.write(rbd_ref)

# Device
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# Load ESM-2 650M (same as rabies)
model, alphabet = esm.pretrained.esm2_t33_650M_UR50D()
model = model.eval().to(device)
batch_converter = alphabet.get_batch_converter()
REPR_LAYER = 33

# Embed full RBD
_, _, tokens = batch_converter([("RBD", rbd_ref)])
tokens = tokens.to(device)
with torch.no_grad():
    out = model(tokens, repr_layers=[REPR_LAYER])
rbd_emb = out["representations"][REPR_LAYER][0, 1:len(rbd_ref)+1].cpu().numpy()
print("RBD embedding:", rbd_emb.shape)

np.save("../data/SARS2_RBD/esm_cache/RBD_esm_650M.npy", rbd_emb)
print("Saved: RBD_esm_650M.npy")

RBD length: 201
Device: cuda
RBD embedding: (201, 1280)
Saved: RBD_esm_650M.npy


In [18]:
# Map each mutation's site to 0-indexed position in rbd_ref
# site 331 → position 0, site 531 → position 200
df_s['seq_idx'] = df_s['site'] - 331

# Verify all indices are in range
print("seq_idx range:", df_s['seq_idx'].min(), "to", df_s['seq_idx'].max())
assert df_s['seq_idx'].min() >= 0
assert df_s['seq_idx'].max() <= 200

# Row-level G-ESM feature
X_g = rbd_emb[df_s['seq_idx'].values]
print("G-ESM feature shape:", X_g.shape)

seq_idx range: 0 to 200
G-ESM feature shape: (23365, 1280)


In [19]:
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier
from scipy import stats
import json, warnings
warnings.filterwarnings('ignore')

# Load features and embeddings
df_s = pd.read_csv("../data/SARS2_RBD/starr_vir_features.csv")
rbd_emb = np.load("../data/SARS2_RBD/esm_cache/RBD_esm_650M.npy")
df_s['label'] = (df_s['escape_mean'] > 0.5).astype(int)
df_s['seq_idx'] = df_s['site'] - 331

y = df_s['label'].values
abs_ = sorted(df_s['antibody'].unique())
print(f"Rows: {len(df_s)}, Antibodies: {len(abs_)}, Positive rate: {y.mean():.4f}")

# Features
df_s['wt_mut'] = df_s['wildtype'] + df_s['mutant']
enc = OneHotEncoder(sparse_output=False, handle_unknown='ignore', dtype=np.float32)
X_cat = enc.fit_transform(df_s[['wt_mut']]).astype(np.float32)
X_pos = df_s[['seq_idx']].values.astype(np.float32)
X_g   = rbd_emb[df_s['seq_idx'].values].astype(np.float32)

def xgb():
    return XGBClassifier(n_estimators=200, max_depth=3, learning_rate=0.1,
        eval_metric='logloss', random_state=42, n_jobs=2, tree_method='hist',
        max_bin=128, subsample=0.8, colsample_bytree=0.5)

def loao(X):
    out = {}
    for a in abs_:
        te = (df_s['antibody'] == a).values; tr = ~te
        if len(np.unique(y[te])) < 2: continue
        s = xgb().fit(X[tr], y[tr]).predict_proba(X[te])[:, 1]
        out[a] = {'auroc': roc_auc_score(y[te], s),
                  'auprc': average_precision_score(y[te], s)}
    return out

def mean_auroc(r): return float(np.mean([r[a]['auroc'] for a in r]))

# --- MODEL A: baseline ---
resA = loao(np.hstack([X_cat, X_pos]))
print(f"A) One-hot + position:  AUROC={mean_auroc(resA):.4f}")

# --- MODEL B: + G-ESM ---
resB = loao(np.hstack([X_cat, X_pos, X_g]))
print(f"B) A + G-ESM:           AUROC={mean_auroc(resB):.4f}")

# --- PER-SITE baseline ---
def persite(tr_mask, te_mask):
    m = df_s[tr_mask].groupby('seq_idx')['label'].mean()
    return df_s.loc[te_mask,'seq_idx'].map(m).fillna(y[tr_mask].mean()).values

resP = {}
for a in abs_:
    te = (df_s['antibody'] == a).values; tr = ~te
    s = persite(tr, te)
    resP[a] = {'auroc': roc_auc_score(y[te], s), 'auprc': average_precision_score(y[te], s)}
print(f"Per-site baseline:      AUROC={mean_auroc(resP):.4f}")

# --- CONTROL E: shuffled G ---
perm = np.random.RandomState(0).permutation(len(rbd_emb))
X_shuf = rbd_emb[perm][df_s['seq_idx'].values]
resE = loao(np.hstack([X_cat, X_pos, X_shuf]))
print(f"E) Shuffled G:          AUROC={mean_auroc(resE):.4f}")

# --- CONTROL F: random vectors ---
X_rand = np.random.RandomState(1).randn(*rbd_emb.shape).astype(np.float32)[df_s['seq_idx'].values]
resF = loao(np.hstack([X_cat, X_pos, X_rand]))
print(f"F) Random vectors:      AUROC={mean_auroc(resF):.4f}")

# --- Paired comparisons ---
def paired(r1, r2):
    x = [r1[a]['auroc'] for a in r1]; z = [r2[a]['auroc'] for a in r2]
    return {'delta': float(np.mean(x)-np.mean(z)),
            'n_higher': int(sum(i>j for i,j in zip(x,z))),
            'wilcoxon_p': float(stats.wilcoxon(x,z).pvalue)}

print("\nB vs A:", paired(resB, resA))
print("B vs per-site:", paired(resB, resP))
print("B vs shuffled:", paired(resB, resE))
print("B vs random:", paired(resB, resF))

Rows: 23365, Antibodies: 12, Positive rate: 0.0307
A) One-hot + position:  AUROC=0.6200
B) A + G-ESM:           AUROC=0.6171
Per-site baseline:      AUROC=0.6134
E) Shuffled G:          AUROC=0.6400
F) Random vectors:      AUROC=0.6074

B vs A: {'delta': -0.0028485935799922313, 'n_higher': 5, 'wilcoxon_p': 1.0}
B vs per-site: {'delta': 0.003733499850220756, 'n_higher': 6, 'wilcoxon_p': 0.7333984375}
B vs shuffled: {'delta': -0.022861398433852043, 'n_higher': 6, 'wilcoxon_p': 0.4697265625}
B vs random: {'delta': 0.009706920076859138, 'n_higher': 8, 'wilcoxon_p': 0.4697265625}


In [20]:
for thr_name, thr_val in [("0.5 (current)", 0.5), 
                           ("top10pct", df_s['escape_mean'].quantile(0.9)),
                           ("top5pct", df_s['escape_mean'].quantile(0.95))]:
    y_test = (df_s['escape_mean'] > thr_val).astype(int).values
    if y_test.mean() < 0.01 or y_test.mean() > 0.5:
        print(f"{thr_name}: skipped (positive rate {y_test.mean():.4f})")
        continue
    # rerun A and B with this label
    # ... (adapt the loao function to take custom labels)
    print(f"{thr_name}: positive rate {y_test.mean():.4f}")

0.5 (current): positive rate 0.0307
top10pct: positive rate 0.0998
top5pct: positive rate 0.0500


In [21]:
for thr_name, thr_val in [("top10pct", df_s['escape_mean'].quantile(0.9)),
                           ("top5pct", df_s['escape_mean'].quantile(0.95))]:
    print(f"\n=== {thr_name} (threshold={thr_val:.4f}) ===")
    y_thr = (df_s['escape_mean'] > thr_val).astype(int).values
    print(f"Positive rate: {y_thr.mean():.4f}")
    
    # Rerun A, B, per-site, shuffled, random with y_thr
    # ... (adapt loao to take y as arg)


=== top10pct (threshold=0.0485) ===
Positive rate: 0.0998

=== top5pct (threshold=0.0649) ===
Positive rate: 0.0500
